# Run whitebox QA and collect taxonomy reports

Runs Testable whitebox for selected branches, exports HTML, saves under
`taxonomy_reports/<L2 group>/<BRANCH>/<BRANCH>_<timestamp>.html` on **main**, and verifies results.

## Parameters

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "lib" / "sa_qa.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from lib import metrics
from lib.registry import load_registry, iter_branches
from lib.sa_qa import run_taxonomy_batch, verify_taxonomy_file, load_env
from lib.qa_reports import save_html_on_main, git_commit_report

ENV_FILE = str(ROOT / ".env.local")
VERSION = "2.6"
REFRESH_BRANCHES = True
SKIP_VERIFY = False
COMMIT_EACH_REPORT = True

# --- Scope: single metric × 4 branch types (override for partial runs) ---
BRANCHES = [
    "SA_DOV_Bug_2.6",
    "SA_DOV_BugFX_2.6",
    "SA_DOV_TCC_2.6",
    "SA_DOV_CC_2.6",
]

# Set USE_MANIFEST=True to run all branches from build_manifest.json instead
USE_MANIFEST = False
MANIFEST = ROOT / "build_manifest.json"

if USE_MANIFEST and MANIFEST.is_file():
    branches = json.loads(MANIFEST.read_text(encoding="utf-8")).get("branches", [])
    print(f"Loaded {len(branches)} branches from build_manifest.json")
else:
    branches = list(BRANCHES)
    print(f"Planned {len(branches)} branches: {', '.join(branches)}")
branches_csv = ",".join(branches)

## Run QA batch

In [ ]:
rc, batch_dir = run_taxonomy_batch(
    env_file=ENV_FILE,
    branches_csv=branches_csv,
    refresh_branches=REFRESH_BRANCHES,
    export_html=True,
    html_only=False,
)
if rc != 0:
    raise RuntimeError(f"Batch failed rc={rc}")
print("Batch dir:", batch_dir)

## Save reports on main + verify

In [ ]:
import os
from lib.metrics import infer_from_branch_name

summary = []
for name in sorted(os.listdir(batch_dir)):
    folder = os.path.join(batch_dir, name)
    if not os.path.isdir(folder):
        continue
    html_files = [f for f in os.listdir(folder) if f.endswith(".html")]
    if not html_files:
        summary.append({"branch": name, "status": "no-html"})
        continue
    html_path = os.path.join(folder, html_files[0])
    branch = name.split("_202")[0] if "_202" in name else name.rsplit("_", 2)[0]
    tech, metric, btype, _ = infer_from_branch_name(branch)
    with open(html_path, encoding="utf-8") as fh:
        html = fh.read()
    full, rel = save_html_on_main(str(ROOT), branch, html)
    commit_sha = ""
    if COMMIT_EACH_REPORT:
        commit_sha = git_commit_report(str(ROOT), rel, branch)
    ok, detail = (True, "skipped") if SKIP_VERIFY else verify_taxonomy_file(full, metric, btype)
    summary.append({"branch": branch, "path": rel, "verify": ok, "commit": commit_sha})
    print(branch, "OK" if ok else "FAIL", rel)

print(f"\nProcessed {len(summary)} report folders")

## Summary table

In [ ]:
for row in summary:
    print(row)